# Compute

In [ ]:
import os
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov
import logging
from datetime import datetime
from joblib import Parallel, delayed

# Setup logging configuration
def setup_logger(sample_name=None):
    """Configure the logger"""
    logger = logging.getLogger('MetaCellPipeline')
    logger.setLevel(logging.DEBUG)
    
    # Clear previous handlers to avoid duplicate logs
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    
    # Console output format
    console_handler = logging.StreamHandler()
    console_format = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(console_format)
    
    # File output format (including sample name)
    os.makedirs('./process/logs/', exist_ok=True)
    log_file = f'./process/logs/pipeline_{sample_name if sample_name else "global"}.log'
    file_handler = logging.FileHandler(log_file, mode='w')
    file_format = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(file_format)
    
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger

# Initialize global logger
global_logger = setup_logger()
global_logger.info("Starting MetaCell processing pipeline")

# Data Loading
try:
    global_logger.info("Loading RNA data...")
    rna = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/adata.h5ad')
    global_logger.info(f"RNA data loaded: {rna.shape[0]} cells, {rna.shape[1]} features")
    
    global_logger.info("Loading ATAC data...")
    atac = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/atac/data_new.h5ad')
    global_logger.info(f"ATAC data loaded: {atac.shape[0]} cells, {atac.shape[1]} features")
except Exception as e:
    global_logger.error(f"Data loading failed: {str(e)}")
    raise

# Merge Data
try:
    global_logger.info("Merging RNA and ATAC data...")
    adata = sc.concat([rna, atac], axis=1)
    adata.layers['counts'] = adata.X
    adata.obs = rna.obs
    del rna
    del atac
    global_logger.info(f"Merged data shape: {adata.shape[0]} cells, {adata.shape[1]} features")
except Exception as e:
    global_logger.error(f"Data merging failed: {str(e)}")
    raise

# Add Dimensionality Reduction Results and Annotations
try:
    global_logger.info("Adding spectral embedding...")
    spectral = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
    adata.obsm['X_spectral'] = spectral
    global_logger.info(f"Spectral embedding added: {spectral.shape}")
    
    global_logger.info("Adding cell type annotations...")
    ann = pd.read_csv('/path/to/hemelas_project/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
    adata.obs['l3'] = ann.loc[adata.obs_names, 'l3'].values
    global_logger.info(f"Added {adata.obs['l3'].nunique()} unique cell types")
except Exception as e:
    global_logger.error(f"Annotation addition failed: {str(e)}")
    raise

# Create Output Directories
os.makedirs('./process/model/', exist_ok=True)
os.makedirs('./process/ad/', exist_ok=True)
os.makedirs('./process/cell_assign/', exist_ok=True)
os.makedirs('./process/logs/', exist_ok=True)

# Sample Processing Function
def process_sample(sample):
    """Complete pipeline for processing a single sample"""
    sample_logger = setup_logger(sample_name=sample)
    sample_logger.info(f"Starting processing for sample: {sample}")
    
    try:
        # 1. Create data subset
        sample_logger.info("Creating subset...")
        subset = adata[adata.obs['sample'] == sample]
        sample_logger.info(f"Subset created: {subset.shape[0]} cells")
        
        # 2. Initialize MetaCell
        sample_logger.info("Initializing MetaCell...")
        meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu'
        )
        
        # 3. Train model
        sample_logger.info("Training MetaCell model...")
        meta_obj.initialize_archetypes()
        meta_obj.train(min_iter=10, max_iter=50)
        sample_logger.info("Model training completed")
        
        # 4. Save model
        model_path = os.path.join('./process/model/', f"{sample}.pkl")
        meta_obj.save(model_path)
        sample_logger.info(f"Model saved to {model_path}")
        
        # 5. Predict metacells
        sample_logger.info("Predicting metacells...")
        ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')
        
        # 6. Calculate metrics
        sample_logger.info("Calculating separation and compactness...")
        ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
        ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()
        
        # 7. Save results
        ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
        ad.write_h5ad(ad_path)
        sample_logger.info(f"Anndata saved to {ad_path}")
        
        assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
        meta_obj.adata.obs[['SEACell']].to_csv(assign_path)
        sample_logger.info(f"Cell assignments saved to {assign_path}")
        
        # 8. Clean up resources
        torch.cuda.empty_cache()
        sample_logger.info(f"Completed processing for sample: {sample}")
        
        return True
    except Exception as e:
        sample_logger.error(f"Error processing sample {sample}: {str(e)}", exc_info=True)
        return False

# Main Execution Flow
if __name__ == '__main__':
    try:
        global_logger.info("Starting sample processing...")
        sample_counts = pd.DataFrame(adata.obs['sample'].value_counts().sort_values(ascending=True))
        global_logger.info(f"Found {len(sample_counts)} samples to process")
        
        # Parallel processing (adjust n_jobs based on server cores)
        results = Parallel(n_jobs=10)(
            delayed(process_sample)(sample) 
            for sample in sample_counts.index
        )
        
        # Calculate statistics
        success_count = sum(results)
        global_logger.info(
            f"Processing completed. Success: {success_count}/{len(sample_counts)} "
            f"({success_count/len(sample_counts):.1%})"
        )
        
        # Generate summary report
        report_path = './process/logs/summary_report.txt'
        with open(report_path, 'w') as f:
            f.write(f"MetaCell Processing Summary\n{'='*30}\n")
            f.write(f"Timestamp: {datetime.now()}\n")
            f.write(f"Total samples: {len(sample_counts)}\n")
            f.write(f"Successful: {success_count}\n")
            f.write(f"Failed: {len(sample_counts) - success_count}\n")
            f.write("\nSample Details:\n")
            for sample, status in zip(sample_counts.index, results):
                f.write(f"{sample}: {'SUCCESS' if status else 'FAILED'}\n")
        
        global_logger.info(f"Summary report saved to {report_path}")
        
    except Exception as e:
        global_logger.error(f"Fatal error in main pipeline: {str(e)}", exc_info=True)
        raise

## 'Cheong_Cell_2023_1-jcov124' The number of cells is 665, an error is reported, and it has been solved

In [ ]:
import os
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov
import logging
from datetime import datetime
from joblib import Parallel, delayed

In [ ]:
rna = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/adata.h5ad')

In [ ]:
rna

In [ ]:
atac = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/atac/data_new.h5ad')

In [ ]:
import numpy as np
from scipy import sparse

def calculate_sparsity(adata):
    if sparse.issparse(adata.X):
        sparsity = 1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    else:
        sparsity = np.mean(adata.X == 0)
    return sparsity

sparsity = calculate_sparsity(rna)
print(f"Sparsity: {sparsity:.2%}")  

In [ ]:
sparsity = calculate_sparsity(atac)
print(f"Sparsity: {sparsity:.2%}")  

In [ ]:
adata = sc.concat([rna, atac], axis=1)

In [ ]:
adata.layers['counts'] = adata.X
adata.obs = rna.obs

In [ ]:
rna = adata

In [ ]:
del adata
del atac

In [ ]:
spectral = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
rna.obsm['X_spectral'] = spectral
    
ann = pd.read_csv('/path/to/hemelas_project/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
rna.obs['l3'] = ann.loc[rna.obs_names, 'l3'].values

In [ ]:
sample_counts = pd.DataFrame(rna.obs['sample'].value_counts().sort_values(ascending=True))

In [ ]:
sample_counts

In [ ]:
sample = 'Cheong_Cell_2023_1-jcov124'

In [ ]:
subset = rna[rna.obs['sample'] == sample]

In [ ]:
subset

In [ ]:
subset.obs['l3'].value_counts()

In [ ]:
meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu',
            n_metacells=9  
        )

In [ ]:
meta_obj.initialize_archetypes()
meta_obj.train(min_iter=10, max_iter=50)

In [ ]:
model_path = os.path.join('./process/model/', f"{sample}.pkl")
meta_obj.save(model_path)

In [ ]:
ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')

In [ ]:
ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()

In [ ]:
ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
ad.write_h5ad(ad_path)

In [ ]:
assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
meta_obj.adata.obs[['SEACell']].to_csv(assign_path)

## 'Thomson_Nat_Immunol_2024-B065_P1_PB00593_04' reported an error when calculating separation, which has been solved

In [ ]:
sample = 'Thomson_Nat_Immunol_2024-B065_P1_PB00593_04'

In [ ]:
subset = rna[rna.obs['sample'] == sample]

In [ ]:
meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu'
        )

In [ ]:
model_path = os.path.join('./process/model/', f"{sample}.pkl")
meta_obj.load(model_path)

In [ ]:
ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')

In [ ]:
ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()

In [ ]:
ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
ad.write_h5ad(ad_path)

In [ ]:
assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
meta_obj.adata.obs[['SEACell']].to_csv(assign_path)

# Merge and analysis

## h5ad concat

In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd

adata_list = []
folder_path  = './process/ad/'

In [ ]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".h5ad"):
        base_name = file_name.replace(".h5ad", "")
        print(f"Processing file: {file_name}")
        adata = sc.read_h5ad(os.path.join(folder_path, file_name))
        adata.obs_names = base_name + '@' + adata.obs_names
        adata_list.append(adata)

In [ ]:
merged_adata = sc.concat(adata_list, join="outer")
merged_adata

In [ ]:
merged_adata.write('./process/adata.h5ad')

### split rna

In [ ]:
combined_genes = merged_adata.var_names[:36601].tolist() + \
                 merged_adata.var_names[[564058, 564059]].tolist()

In [ ]:
rna = merged_adata[:,:36601]
rna

In [ ]:
import numpy as np
from scipy import sparse

def calculate_sparsity(adata):
    if sparse.issparse(adata.X):
        sparsity = 1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    else:
        sparsity = np.mean(adata.X == 0)
    return sparsity

sparsity = calculate_sparsity(rna)
print(f"Sparsity: {sparsity:.2%}") 

In [ ]:
rna.write('./process/rna.h5ad')

### split atac

In [ ]:
merged_adata

In [ ]:
combined_peaks = merged_adata.var_names[36599:564058].tolist()

In [ ]:
len(combined_peaks), combined_peaks[527459-1]

In [ ]:
atac = merged_adata[:, combined_peaks]
atac

In [ ]:
sparsity = calculate_sparsity(atac)
print(f"Sparsity: {sparsity:.2%}")  # For example, "Sparsity: 95.23%"

In [ ]:
atac.write('./process/atac.h5ad')

In [ ]:
merged_adata.obs

## cell_assign

In [ ]:
import os

csv_list = []
folder_path  = './process/cell_assign'

In [ ]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        base_name = file_name.replace(".csv", "")
        print(f"Processing file: {file_name}")
        data = pd.read_csv(os.path.join(folder_path, file_name), index_col=0)
        data['SEACell'] = base_name+'@'+data['SEACell'].str.replace('SEACell-', '')
        csv_list.append(data)

In [ ]:
merged_df = pd.concat(csv_list, axis=0)
merged_df

## umap

In [ ]:
adata = rna

In [ ]:
merged_df = merged_df.reindex(adata.obs_names)
merged_df

In [ ]:
adata.obs['SEACell'] = merged_df['SEACell']

In [ ]:
umap = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
umap

In [ ]:
adata.obsm['X_umap'] = umap

In [ ]:
umap_coords = adata.obsm['X_umap']  # shape: (n_cells, 2)
metacell_labels = adata.obs[['SEACell']]  # metacell

df = pd.DataFrame(umap_coords, columns=['UMAP_1', 'UMAP_2'])
df['metacell'] = metacell_labels['SEACell'].tolist()

metacell_coords = df.groupby('metacell')[['UMAP_1', 'UMAP_2']].mean()

print(metacell_coords)

In [ ]:
metacell_coords.to_csv('./process/umap.csv')

In [ ]:
ad = sc.read_h5ad('./process/rna.h5ad')
ad

In [ ]:
metacell_coords = metacell_coords.reindex(ad.obs_names)
metacell_coords

In [ ]:
ad.obsm['X_umap'] = metacell_coords.to_numpy()

In [ ]:
sc.pl.umap(ad, color='celltype')

In [ ]:
sc.pl.umap(rna,color='l3')

In [ ]:
rna_f = rna[rna.obs['l3'].isin(ad.obs['celltype'])]
rna_f

In [ ]:
sc.pl.umap(ad, color='celltype', palette=rna_f.uns['l3_colors'].tolist(), show=True, save='seacells_cell', size=8)

In [ ]:
sc.pl.umap(ad, color='celltype_purity', show=True, save='_purity')

In [ ]:
ad.obs['total'] = 'total'

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 8))  
sc.pl.violin(ad, keys='compactness', groupby='total', rotation=90, ax=ax, save='_compactness_total')
plt.show()

# Contour

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (6,6))

ov.pl.embedding(ad,
                basis='X_umap',
                color=['celltype'],
                frameon='small',
                show=False,
                ax=ax)

ov.pl.contour(ax=ax,adata=ad,groupby='celltype',clusters=[
'ELP',
'CMP',
'MEMP',
'HSC/MPP',
'Mid erythroid',
'Late erythroid',
'Early erythroid'
],
       basis='X_umap',contour_threshold=0.1,colors='#000000',
        linestyles='dashed',)

plt.savefig('./figures/contour/hspc_ery.pdf', format='pdf', bbox_inches='tight')

# Draw the corresponding relationship between seacells and original cells

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov

In [ ]:
adata = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/adata.h5ad')

In [ ]:
umap = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
adata.obsm['X_umap'] = umap
ann = pd.read_csv('/path/to/hemelas_project/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
adata.obs['l3'] = ann.loc[adata.obs_names, 'l3'].values

In [ ]:
merged_df = merged_df.reindex(adata.obs_names)
merged_df

In [ ]:
merged_df.to_csv('./process/cell_assign_all.csv')

In [ ]:
adata.obs['SEACell'] = merged_df['SEACell']

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,8))
ov.pl.embedding(
    adata,
    basis="X_umap",
    color=['l3'],
    frameon='small',
    title="Meta cells",
    #legend_loc='on data',
    legend_fontsize=14,
    legend_fontoutline=2,
    size=1,
    ax=ax,
    alpha=0.1,
    #legend_loc='', 
    add_outline=False, 
    #add_outline=True,
    outline_color='black',
    outline_width=1,
    show=False,
    #palette=ov.utils.blue_color[:],
    #legend_fontweight='normal'
)
ov.single.plot_metacells(ax,adata,color='#CB3E35', size=1)

plt.savefig('./figures/all_metacells_plot.pdf', format='pdf', bbox_inches='tight')

# Trajectory analysis-CellRank Meets CytoTRACE

### Import packages & data

In [ ]:
import cellrank as cr
import scanpy as sc
import scvelo as scv

cr.settings.verbosity = 2
sc.settings.set_figure_params(frameon=False, dpi=100)

In [ ]:
import warnings

warnings.simplefilter("ignore", category=UserWarning)

In [ ]:
adata = sc.read('./process/hspc/hspc_raw.h5ad')
adata

In [ ]:
adata.X.max()

### Explore the data

This zebrafish dataset contains 12 time-points spanning 3.3 - 12 hours past fertilization. We can use the original force-directed layout to plot cells, colored by developmental stage:

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color="celltype")

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color="CD34")

### Preprocess

Before we can start applying the CytoTRACE kernel, we'll have to do some basic preprocessing of the data. 

In [ ]:
# filter, normalize total counts and log-transform
sc.pp.filter_genes(adata, min_cells=10)
scv.pp.normalize_per_cell(adata, enforce=True)
sc.pp.log1p(adata)

# hvg annotation
sc.pp.highly_variable_genes(adata)
print(f"This detected {adata.var['highly_variable'].sum()} highly variable genes. ")

CytoTRACE uses imputed gene expression to calculate the score of differentiation potential. Here, we use use {func}`scvelo.pp.moments` for imputation. We're copying our {attr}`~anndata.AnnData.X` matrix into the layers because that's where {func}`~scvelo.pp.moments` expects to find counts for imputation. 

In [ ]:
adata.layers["spliced"] = adata.X
adata.layers["unspliced"] = adata.X
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

## Use developmental potential to direct differentiation

### Compute the CytoTRACE score

Import the {class}`~cellrank.kernels.CytoTRACEKernel`, initialize it using the preprocessed {class}`~anndata.AnnData` object and compute the CytoTRACE score: 

In [ ]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata).compute_cytotrace()

We can visually check that our computed CytoTRACE pseudotime correlates well with ground-truth real-time annotations:

In [ ]:
adata

In [ ]:
sc.pl.embedding(
    adata,
    color=["ct_pseudotime", "celltype"],
    basis="X_umap",
    color_map="gnuplot2",
    # save='cytotrace'
)

To make this a bit more quantitative, we can look at the distribution of our CytoTRACE pseudotime over real time points and show this via violin plots:

In [ ]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="celltype", rotation=90)

As expected, the CytoTRACE pseudotime increases for later time points. Thus, we can use this score to bias graph edges into the direction of increasing differentiation status, similar to how we do this for other pseudotimes with the {class}`~cellrank.kernels.PseudotimeKernel`. 

### Compute & visualize a transition matrix

Compute the transition matrix using {meth}`~cellrank.kernels.CytoTRACEKernel.compute_transition_matrix`. 

In [ ]:
ctk.compute_transition_matrix(threshold_scheme="soft", nu=0.5)

To get some intuition for this transition matrix, we can project it into an embedding and draw streamlines - the following plot will look like the plots you're used to from scVelo and velocyto for visualizing RNA velocity {cite}`bergen:20,manno:18`. 

:::{note}

There's **no RNA velocity** here, we're visualizing the directed transition matrix we computed using the kNN graph as well as the CytoTRACE pseudotime.

:::

In [ ]:
ctk.plot_projection(basis="X_umap", color="celltype", legend_loc="right", size=100, 
                    # save='ct'
                   )

Another way to gain intuition for the transition matrix is by drawing some cells from the early stage and to use these as starting cells to simulate random walks on the transition matrix. This visualization technique is less biased by the low-dimensional embedding as the random walks are computed directly based on the transition matrix. 

In [ ]:
ctk.plot_random_walks(
    n_sims=100,
    start_ixs={"celltype": "HSC/MPP"},
    basis="X_umap",
    color="celltype",
    legend_loc="right",
    seed=1,
    # save='cytotrace_walk.pdf'
)

Black dots denote sampled starting cells for random walks, yellow dots denote end cells. We terminated each random walk after a predefined number of steps. Random walks are colored according to how long they've been running for - i.e. the initial segments are more black/blue whereas the late segments are more orange/yellow. We can see that many random walks terminate in one of the two terminal states, as expected. Quite a few of them seem to get stuck early on, this could be further investigated. 

:::{note}

The visualization techniques demonstrated here work for **every kernel**, no matter whether it's a {class}`~cellrank.kernels.VelocityKernel`, {class}`~cellrank.kernels.PseudotimeKernel`, {class}`~cellrank.kernels.CytoTRACEKernel` or some other CellRank {mod}`kernel <cellrank.kernels>`. 

:::

## Closing matters

### What's next? 

In this tutorial, you learned how to use CellRank to compute a transition matrix based on the CytoTRACE pseudotime and how it can be visualized in low dimensions. The real power of CellRank comes in when you use estimators to analyze the transition matrix directly, rather than projecting it. For the next steps, we recommend to:

- go through the {doc}`initial and terminal states tutorial <../estimators/600_initial_terminal>` to learn how to use the transition matrix to automatically identify initial and terminal states. 
- take a look at the {class}`API <cellrank.kernels.CytoTRACEKernel>` to learn about parameter values you can use to adapt these computations to your data. 
- read the original CytoTRACE publication and considering limitations of this appraoch {cite}`gulati:20`. 

### Package versions

In [ ]:
cr.logging.print_versions()

## gene_trends

In [ ]:
import cellrank as cr
import scvelo as scv
import scanpy as sc

In [ ]:
adata

In [ ]:
ptk = cr.kernels.PseudotimeKernel(adata, time_key="ct_pseudotime")
ptk.compute_transition_matrix()

estimator = cr.estimators.GPCCA(ptk)
estimator.compute_schur()

estimator.compute_macrostates(3, cluster_key="celltype")
estimator.predict_terminal_states()
estimator.compute_fate_probabilities()

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","KLF1","EBF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
#    save='gene_trends.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    # save='gene_trends_heatmap_elp.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    # save='gene_trends_heatmap_MEMP.pdf'
)

## accessible trend

In [ ]:
import pandas as pd
import scanpy as sc

In [ ]:
arc = pd.read_csv('./process/atac_seacell/GeneAccessibility_all.csv', index_col=0)

In [ ]:
arc = arc.where(arc >= 0, 0).fillna(0)

In [ ]:
arc

In [ ]:
adata = sc.AnnData(
    X=arc.values,          
    obs=arc.index.to_frame(name='cell_id'),  
    var=arc.columns.to_frame(name='gene_name')  
)

In [ ]:
adata.write_h5ad('./process/arc.h5ad')

In [ ]:
arc_hspc = arc.loc[adata.obs_names,adata.var_names]
arc_hspc

In [ ]:
arc_hspc = arc_hspc.where(arc_hspc >= 0, 0).fillna(0)

In [ ]:
arc_hspc

In [ ]:
adata.X = arc_hspc

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","EBF1","KLF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
    save='gene_trends_atac.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_elp_atac.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_memp_atac.pdf'
)

In [ ]:
scv.pp.normalize_per_cell(adata, enforce=True)
sc.pp.log1p(adata)

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","EBF1","KLF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
    save='gene_trends_atac_norm.pdf'
)

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["CD34"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
#    save='gene_trends.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_elp_atac_norm.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_memep_atac_norm.pdf'
)

# Chromatin potential

## Gene accessibility scores

This notebook is a tutorial for ATAC analysis using SEACells and includes computation of gene-peak associations, ATAC gene scores, gene accessibility scores and identification of highly regulated genes

### Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
import SEACells

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Some plotting aesthetics
%matplotlib inline

sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100


### Load Data

In [ ]:
rna_meta_ad = sc.read('process/rna.h5ad')
atac_meta_ad = sc.read('data/atac_meta_ad.h5ad')

In [ ]:
# # Revise var_names
# def replace_first_dash(s):
#     return s.replace('-', ':', 1)

# atac_meta_ad.var_names = [replace_first_dash(name) for name in atac_meta_ad.var_names]

# # verify
# print(atac_meta_ad.var_names[:5])

In [ ]:
# def _add_atac_meta_data(atac_meta_ad, atac_ad, n_bins_for_gc):
#     atac_ad.var["log_n_counts"] = np.ravel(np.log10(atac_ad.X.sum(axis=0)))

#     atac_meta_ad.var["GC_bin"] = np.digitize(
#         atac_ad.var["GC"], np.linspace(0, 1, n_bins_for_gc)
#     )
#     atac_meta_ad.var["counts_bin"] = np.digitize(
#         atac_ad.var["log_n_counts"],
#         np.linspace(
#             atac_ad.var["log_n_counts"].min(),
#             atac_ad.var["log_n_counts"].max(),
#             n_bins_for_gc,
#         ),
#     )

####  add gc_bin

In [ ]:
# import pandas as pd
# from pyfaidx import Fasta
# from Bio.SeqUtils import gc_fraction as GC

In [ ]:
# def calculate_gc_content(bed_file: str, genome_fasta: str) -> pd.DataFrame:
#     """
#     Calculates the GC content for each region in a BED file and returns a DataFrame with detailed results.
    
#     Args:
#         bed_file (str): Path to the BED file (format: chr start end [optional columns]).
#         genome_fasta (str): Path to the reference genome FASTA file (must be indexed, e.g., with samtools).
    
#     Returns:
#         pd.DataFrame: A table containing chromosome, start, end, sequence, and GC content.
    
#     Example:
#         >>> df = calculate_gc_content("peaks.bed", "hg38.fa")
#         >>> df.head()
#     """
#     try:
#         # Load reference genome
#         genome = Fasta(genome_fasta)
        
#         # Read BED file as DataFrame (automatically handles comment lines and column names)
#         bed_df = pd.read_csv(
#             bed_file, 
#             sep='\t', 
#             comment='#', 
#             header=None,
#             names=['chr', 'start', 'end'] + [f'extra_{i}' for i in range(3, 100)]  # Handle potential extra columns
#         )
        
#         # Initialize results list
#         results = []
        
#         for _, row in bed_df.iterrows():
#             chrom, start, end = row['chr'], int(row['start']), int(row['end'])
            
#             try:
#                 # Extract sequence (BED is 0-based, pyfaidx uses 1-based indexing)
#                 sequence = str(genome[chrom][start+1:end].seq).upper()  # Ensure uppercase
                
#                 # Calculate GC content
#                 gc_percent = GC(sequence)
                
#                 # Save results (including original BED rows and calculated results)
#                 result_row = row.to_dict()
#                 result_row.update({
#                     'sequence': sequence,
#                     'gc_percent': gc_percent,
#                     'length': len(sequence)
#                 })
#                 results.append(result_row)
                
#             except (KeyError, IndexError) as e:
#                 print(f"Warning: Skipping invalid region {chrom}:{start}-{end} (Error: {str(e)})")
#                 continue
                
#         # Convert to DataFrame
#         result_df = pd.DataFrame(results)
        
#         # Reorder columns (place key information at the beginning)
#         base_cols = ['chr', 'start', 'end', 'length', 'gc_percent', 'sequence']
#         other_cols = [col for col in result_df.columns if col not in base_cols]
#         return result_df[base_cols + other_cols]
    
#     except Exception as e:
#         raise RuntimeError(f"Failed to calculate GC content: {str(e)}") from e

In [ ]:
# results = calculate_gc_content('/path/to/hemelas_project/z_others/0_ref_peak/merged_summit/All_Samples.fwp.filter.non_overlapping.bed', 
#                                '/path/to/reference_data/refdata-gex-GRCh38-2020-A/fasta/genome.fa')

In [ ]:
# results.iloc[:,:8].to_csv('./data/peak_gc.csv')

In [ ]:
# result = pd.read_csv('./data/peak_gc.csv', index_col=0)
# result

In [ ]:
# atac_meta_ad.var['GC'] = result['gc_percent'].tolist()

In [ ]:
# atac_meta_ad.var["GC_bin"] = np.digitize(
#     atac_meta_ad.var["GC"], np.linspace(0, 1, 50)
# )

In [ ]:
# atac_meta_ad.var["GC_bin"]

#### add counts_bin

In [ ]:
# sc.pp.calculate_qc_metrics(atac_meta_ad,inplace=True)

In [ ]:
# atac_meta_ad.var["counts_bin"] = np.digitize(
#         atac_meta_ad.var["log1p_total_counts"],
#         np.linspace(
#             atac_meta_ad.var["log1p_total_counts"].min(),
#             atac_meta_ad.var["log1p_total_counts"].max(),
#             50,
#         ),
# )

In [ ]:
# atac_meta_ad.var["counts_bin"]

In [ ]:
# atac_meta_ad.write_h5ad('./data/atac_meta_ad.h5ad')

### Gene-peak correlations

Using the paired multiome metacell data, the next step is to compute the correlation of gene expression and accessbility of peaks within the vicinity of the gene. 

Computation of gene peak correlations requires the following parameters :
1. GTF file with gene annotations. <b> Note: </b> Chromosome names should be numbered 1, 2 and the "chr" prefix will be added by SEACells 
2. Genomic span around genes to test the correlations 

Human GTF file is available at [https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf](https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf) and can be downloaded using:

In [ ]:
#! wget https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf -O data/hg38.gtf

In [ ]:
# In this example, we compute gene peak correlations for the first ten genes since 
# this process is computationally intensive
# gene_set = rna_meta_ad.var_names
# gene_peak_cors = SEACells.genescores.get_gene_peak_correlations(atac_meta_ad, rna_meta_ad, 
#                                            path_to_gtf='data/hg38.gtf', 
#                                            span=100000, 
#                                            n_jobs=100,
#                                            gene_set=gene_set)

In [ ]:
# import pickle

# with open('gene_peak_cors.pkl', 'wb') as f:
#     pickle.dump(gene_peak_cors, f)

#### Highly regulated genes

In [ ]:
gene_peak_cors = pd.read_pickle('./atac_seacell/all/gene_peak_cors.pkl') 

In [ ]:
len(gene_peak_cors)

Highly regulated genes i.e., genes that are correlated with multiple peaks can be identified using the `get_gene_peak_assocations` function. `get_gene_peak_assocations` returns the number of significantly peaks correlated with each gene

In [ ]:
peak_counts = SEACells.genescores.get_gene_peak_assocations(gene_peak_cors, 
                                                           pval_cutoff=1e-1,
                                                           cor_cutoff=0.1)

In [ ]:
peak_counts.value_counts()

In [ ]:
# Plot the distribution to identify genes with higher degree of regulation
plt.scatter(np.arange(len(peak_counts)), 
           np.sort(peak_counts), s=20)
sns.despine()
plt.xlabel('Gene rank')
plt.ylabel('No. of correlated peaks')

#### Gene scores

Gene scores are computed as the weighted sum of the accessiblity of correlated peaks and can be computed using `get_gene_scores`. 

In [ ]:
# gene_scores = SEACells.genescores.get_gene_scores(atac_meta_ad, 
#                                                   gene_peak_cors,
#                                                   pval_cutoff=1e-1,
#                                                   cor_cutoff=0.1)

In [ ]:
gene_scores = pd.read_csv('gene_scores.csv')

In [ ]:
gene_scores.head()

In [ ]:
gene_scores.shape

`gene_scores` is a `pandas.DataFrame` with metacells as rows and genes as columns. This can be used for any downstream analysis such as clustering, visualization etc.

### Gene-accessibility

This section describes how to compute gene accessiblity metrics using SEACell metacells.

#### Open peaks in metacells

The first step is to identify the subset of peaks that are open in each metacell. `determine_metacell_open_peaks` function can be used to determine this. 

By default, all peaks are tested to check if they are open or closed in every metacell. A subset of peaks can be specified using the `peak_set` parameter. 

This function also requires a low-dimensional embedding such as `X_svd`. We can summarize the SVD of single-cell ATAC for this analysis

In [ ]:
atac_meta_ad

In [ ]:
atac_meta_ad.obs['n_counts'] = atac_meta_ad.obs['total_counts']

In [ ]:
# read spectral

In [ ]:
spectral = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
spectral

In [ ]:
cell_assign = pd.read_csv('./process/cell_assign_all.csv', index_col=0)
cell_assign

In [ ]:
sc_spectral = pd.DataFrame(spectral, index=cell_assign.index)

In [ ]:
atac_meta_ad.obsm['X_spectral'] = sc_spectral.groupby(cell_assign['SEACell']).mean().loc[atac_meta_ad.obs_names, :]

In [ ]:
atac_meta_ad

In [ ]:
# # We will reuse the atac_meta_ad computed above

# # Add SVD summary to atac meta ad
# seacell_label = 'SEACell'
# sc_svd = pd.DataFrame(atac_ad.obsm['X_svd'], index=atac_ad.obs_names)
# atac_meta_ad.obsm['X_svd'] = sc_svd.groupby(atac_ad.obs[seacell_label]).mean().loc[atac_meta_ad.obs_names, :]

In [ ]:
# # Determine open peaks in each metacell
# SEACells.accessibility.determine_metacell_open_peaks(atac_meta_ad, peak_set=None, low_dim_embedding='X_spectral', pval_cutoff=1e-2,
#                                   read_len=147, n_neighbors=3, n_jobs=1)
# # This function will add 'OpenPeaks' to the Anndata layers and is a binary matrix 
# # indicating whether the peak is open or closed in the metacell

In [ ]:
# np.save('OpenPeaks.npy', atac_meta_ad.layers['OpenPeaks'])

In [ ]:
atac_meta_ad.layers['OpenPeaks'] = np.load('./atac_seacell/all/OpenPeaks.npy')

#### Gene accessibility metric

Open peaks are used to compute gene accessiblity metric which represents the fraction of correlated open peaks. 

<b>Warning: </b> This metric is only reliable if there are sufficient number of open peaks associated with each gene. 
It is recommended to be used for only genes with high regulation

In [ ]:
# Use the highly regulated genes as the gene set of interest 
# Plot the distribution to identify genes with higher degree of regulation
plt.scatter(np.arange(len(peak_counts)), 
           np.sort(peak_counts), s=20)
sns.despine()
plt.xlabel('Gene rank')
plt.ylabel('No. of correlated peaks')

In [ ]:
# Select genes based on the elbow point
high_reg_genes = peak_counts.index[peak_counts > 9]

In [ ]:
high_reg_genes

In [ ]:
gene_peak_cors

In [ ]:
# # Compute gene accessibility
SEACells.accessibility.get_gene_accessibility(atac_meta_ad, gene_peak_cors, 
                                              gene_set=peak_counts.index, pval_cutoff=1e-1, cor_cutoff=0.1)
# p-value and corrrelation cutoffs are used for correlated peaks
# # This function will add 'GeneAccessibility' to the Anndata `.obsm` field

In [ ]:
atac_meta_ad.obsm['GeneAccessibility']

In [ ]:
atac_meta_ad.obsm['GeneAccessibility'].max()

In [ ]:
atac_meta_ad.obsm['GeneAccessibility'].to_csv('./GeneAccessibility_all.csv')

Gene accessiblity metrics can be used as inputs for downstream analyses.

#### Visualization

In [ ]:
rna_meta_ad.obs

In [ ]:
umap = pd.read_csv('./process/umap.csv',index_col=0)
umap = umap.reindex(rna_meta_ad.obs_names)
umap

In [ ]:
rna_meta_ad.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna_meta_ad, color='celltype')

In [ ]:
GeneAccessibility = pd.read_csv('./GeneAccessibility_all.csv', index_col=0)
GeneAccessibility

In [ ]:
genes = ['KLF1', 'GATA1', 'SPI1']

In [ ]:
sc.pl.scatter(rna_meta_ad, basis='umap', color=genes)

In [ ]:
temp = rna_meta_ad[:, genes]

In [ ]:
temp

In [ ]:
temp.obs[genes] = GeneAccessibility[genes]

In [ ]:
temp.obs

In [ ]:
sc.pl.scatter(temp, basis='umap', color=genes)

## Potential

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cdist

In [ ]:
adata = sc.read_h5ad('./process/chrom_pote.h5ad')
adata

In [ ]:
adata.obsm['dorc_scores']

In [ ]:
adata.obsm['gene_expression']

In [ ]:
spectral = pd.read_csv('./process/spectral.csv', index_col=0).loc[adata.obs_names,:]
spectral

In [ ]:
adata.obsm['X_cisTopic'] = spectral.to_numpy()

In [ ]:
adata.layers['RNA'] = adata.X

In [ ]:
adata.obsm['dorc_scores'] = adata.obsm['dorc_scores']

In [ ]:
sc.pl.umap(adata, color='celltype')

In [ ]:
adata

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import identity
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

def compute_chromatin_potential(adata, 
                               k_smooth=50, 
                               k_neighbors=10,
                               smooth_nn=15,
                               n_grid=40):
    """Main computation function: Chromatin potential calculation workflow."""
    # 1. Data Smoothing
    sc.pp.neighbors(adata, n_neighbors=k_smooth, use_rep='X_cisTopic')
    adj_matrix = adata.obsp['connectivities'].copy()
    adj_matrix += identity(adj_matrix.shape[0])
    row_sum = adj_matrix.sum(axis=1).A1
    norm_adj = adj_matrix / row_sum[:, None]
    
    # Smooth DORC and gene expression
    adata.obsm['dorc_smoothed'] = norm_adj.dot(adata.obsm['dorc_scores'])
    adata.obsm['gene_smoothed'] = norm_adj.dot(adata.obsm['gene_expression'])
    
    # 2. Find RNA Neighbors
    scaler = StandardScaler()
    dorc_scaled = scaler.fit_transform(adata.obsm['dorc_smoothed'])
    gene_scaled = scaler.fit_transform(adata.obsm['gene_smoothed'])
    corr_matrix = np.dot(dorc_scaled, gene_scaled.T) / (dorc_scaled.shape[1]-1)
    top10_indices = np.argsort(-corr_matrix, axis=1)[:, :k_neighbors]
    
    # 3. Calculate Direction Vectors (in UMAP space)
    X_umap = adata.obsm['X_umap']
    target_positions = np.array([X_umap[indices].mean(axis=0) for indices in top10_indices])
    directions = target_positions - X_umap
    distances = np.linalg.norm(directions, axis=1)
    
    # Normalize and store
    adata.obs['chromatin_potential'] = distances / distances.max()
    adata.obsm['raw_directions'] = directions
    
    # 4. Direction Smoothing
    sc.pp.neighbors(adata, n_neighbors=smooth_nn, use_rep='X_umap')
    adj_smooth = adata.obsp['connectivities']
    row_sum_smooth = adj_smooth.sum(axis=1).A1
    norm_adj_smooth = adj_smooth / row_sum_smooth[:, None]
    adata.obsm['smoothed_directions'] = norm_adj_smooth.dot(directions)
    
    # 5. Generate Valid Grid
    generate_valid_grid(adata, n_grid)
    
def generate_valid_grid(adata, n_grid=40):
    """Generate a grid system with valid direction filtering."""
    umap_coords = adata.obsm['X_umap']
    
    # Create grid
    x_min, x_max = umap_coords[:,0].min()-0.1, umap_coords[:,0].max()+0.1
    y_min, y_max = umap_coords[:,1].min()-0.1, umap_coords[:,1].max()+0.1
    grid_x = np.linspace(x_min, x_max, n_grid)
    grid_y = np.linspace(y_min, y_max, n_grid)
    grid_points = np.array(np.meshgrid(grid_x, grid_y)).T.reshape(-1,2)
    
    # Mark valid grid points (grid points containing cells)
    nn = NearestNeighbors(n_neighbors=1).fit(grid_points)
    _, indices = nn.kneighbors(umap_coords)
    grid_counts = np.bincount(indices.flatten(), minlength=len(grid_points))
    valid_mask = grid_counts > 0
    
    # Calculate directions for valid grid points
    valid_directions = []
    for i in np.where(valid_mask)[0]:
        cell_mask = (indices.flatten() == i)
        avg_dir = adata.obsm['smoothed_directions'][cell_mask].mean(axis=0)
        valid_directions.append(avg_dir)
    
    adata.uns['grid_velocity'] = {
        'grid_points': grid_points[valid_mask],
        'directions': np.array(valid_directions)
    }

def plot_chromatin_potential(adata, 
                            scale_factor=50, 
                            grid_scale=80,
                            plot_cells=True,
                            plot_grid=True,
                            save=True,
                            save_path="chromatin_potential.pdf",
                            dpi=300):
    """Visualize chromatin potential and support PDF export."""
    import os
    
    plt.figure(figsize=(10,6), dpi=dpi)
    
    # Plot cell type background
    sc.pl.embedding(adata, basis='X_umap', color='celltype', 
                   alpha=0.3, show=False, frameon=False, title='')
    
    # # Plot cell-level directions
    # if plot_cells:
    #     plt.quiver(adata.obsm['X_umap'][:,0], adata.obsm['X_umap'][:,1],
    #               adata.obsm['smoothed_directions'][:,0],
    #               adata.obsm['smoothed_directions'][:,1],
    #               scale=scale_factor, width=0.002, 
    #               color='grey', alpha=0.5, headlength=3)
    
    # Plot grid-level directions
    if plot_grid and 'grid_velocity' in adata.uns:
        gp = adata.uns['grid_velocity']['grid_points']
        gd = adata.uns['grid_velocity']['directions']
        plt.quiver(gp[:,0], gp[:,1], gd[:,0], gd[:,1],
                  scale=grid_scale, width=0.004, 
                  color='black', headlength=5, headaxislength=4)
    
    plt.title(f"Chromatin Potential (Scale: {scale_factor}/{grid_scale})")
    plt.tight_layout()
    
    # Save PDF
    if save:
        plt.savefig(save_path, bbox_inches='tight', dpi=dpi)
        print(f"PDF saved to: {os.path.abspath(save_path)}")
    
    plt.show()

# Execute complete workflow
compute_chromatin_potential(adata, n_grid=40)
plot_chromatin_potential(adata, scale_factor=60, grid_scale=100)